# Cell Tracking — TemporalUNet3D + SimpleNodeTransformer

Mirrors the structure of `nnUNet.ipynb` but for the tracking task.

Pipeline:
1. **Data** — zarr images + geff tracks
2. **Dataset** — `TrackDataset`: sliding windows of W consecutive frames
3. **Model** — `UNetNodeTransformer`: UNet features + detection head + cross-attention edge predictor
4. **Inference** — load checkpoint, detect peaks, show results

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
cd ../src

In [ ]:
import glob
import json
import torch
import zarr
import polars as pl
import numpy as np
import tracksdata as td
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pathlib import Path
from collections import Counter

In [ ]:
from tracking.dataset import TrackDataset
from tracking.data import load_dataset_windows

from tracking.targets import detect_and_match, build_matched_edge_targets, pos_embed_torch
from tracking.losses import TrackingLoss, detection_loss, edge_loss
from tracking.post_process import detect_peaks
from tracking.plot import plot_detection, plot_edges, plot_sample

from model_zoo.models_track import define_track_model, UNetNodeTransformer
from tracking_cellmot.io import open_dataset, list_datasets


In [ ]:
from params import TRAIN_DIR, LOG_FOLDER, SCALE

TRAIN_DIR = Path(TRAIN_DIR)
LOG_FOLDER = Path(LOG_FOLDER)

DOWNSAMPLE = (1, 1, 1)   # (Z, Y, X) strides used at training time
WINDOW_SIZE = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Data

Each sample is a pair of files:
- `.zarr` — 4D image volume `(T, Z, Y, X)` uint16
- `.geff` — cell tracks (node coordinates + edges between consecutive frames)

In [ ]:
all_paths = list_datasets(TRAIN_DIR, require_geff=True)

embryos = [p.name.split('_')[0] for p in all_paths]
Counter(embryos)

## 2. Dataset — what gets fed to the model

`TrackDataset` produces fixed-size windows of `W` consecutive frames:
- `imgs`: `(W, Z_ds, Y_ds, X_ds)` float16 — normalised, spatially downsampled
- `coords`: `(W, max_nodes, 3)` — GT node positions in downsampled space
- `masks`: `(W, max_nodes)` bool — which slots are real nodes
- `targets`: `(W-1, max_nodes, max_nodes)` — GT transition matrices

In [ ]:
ds_path = np.random.choice(all_paths)
ds_path.name

In [ ]:
vm, windows = load_dataset_windows(
    ds_path, window_size=WINDOW_SIZE,
)

print(vm)

In [ ]:
win_idx = max(range(len(windows)), key=lambda i: sum(windows[i].node_counts))  # max frames number
w = windows[win_idx]

In [ ]:
ds = TrackDataset(
    [(vm, windows)],
    augment=True,
    crop_size=None,  # (64, 128, 128)
    weak_positives_cfg={
        "max_nodes": 512,
        "use": True,
        "threshold": 1,
        "nms_um": 5.0,
    },
    brightness_strength=1,
)

In [ ]:
# from torch.utils.data import DataLoader
# from tqdm import tqdm

# dataloader = DataLoader(ds, batch_size=8, shuffle=False)

# for batch in tqdm(dataloader):
#     print(batch['weak_positives'].shape)
#     # break

In [ ]:
%%time
sample = ds[win_idx]

print("\nSample keys/shapes:")
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:12s}  {tuple(v.shape)}  {v.dtype}")

In [ ]:
sample = ds[win_idx]

zs = np.unique(sample["coords"].numpy()[:, :, 0]).astype(int)
z = np.random.choice(zs)
# z = zs[0]

plot_sample(sample, t_start=windows[win_idx].t_start, tol=3, z=z)

## 3. Model

`UNetNodeTransformer` = `TemporalUNet3D` + detection head + `SimpleNodeTransformer`

1. **encode(imgs)** → UNet features `(B, W, C, Z_ds, Y_ds, X_ds)` + detection logits `(B, 1, Z_ds, Y_ds, X_ds)` per frame
2. **detect_and_match** → detected peak coords + sinusoidal pos-embeddings
3. **_index_features** → UNet features at detected positions
4. **predict_edges** → cross-attention transformer → `(B, N_src, N_tgt)` edge logits

In [ ]:
from model_zoo.models_track import define_track_model

In [ ]:
model = define_track_model(
    unet_out_channels=384,
    unet_layers=[32, 64, 128],
    transformer_hidden=384,
    transformer_heads=8,
    transformer_blocks=8,
    unet_pool_stride=(1, 2, 2),
    unet_n_decode_steps=1,
    verbose=1
).to(device)

In [ ]:
# weights = torch.load("/data/kaggle-cell-tracking/logs/2026-08-13/25/fold_0/ckpt_08000.pt")
# # weights = torch.load("/data/kaggle-cell-tracking/logs/2026-08-13/7/fold_0/ckpt_final.pt")
# model.load_state_dict(weights['model_state_dict'])


In [ ]:
print(f"Total parameters : {model.count_parameters() / 1e6:.2f}M")
print(f"  UNet           : {sum(p.numel() for p in model.unet.parameters()) / 1e6:.2f}M")
print(f"  Detect head    : {sum(p.numel() for p in model.detect_head.parameters()) / 1e6:.2f}M")
print(f"  Transformer    : {sum(p.numel() for p in model.transformer.parameters()) / 1e6:.2f}M")

In [ ]:
sample = ds[win_idx]
imgs_batch = sample["imgs"].float().unsqueeze(0).to(device)

bs, window_size, d, h, w = imgs_batch.shape
print('Input shape:', imgs_batch.shape)

coords = sample["coords"].to(device).unsqueeze(0)
masks = sample["masks"].to(device).unsqueeze(0)
targets = sample["targets"].to(device).unsqueeze(0)

image_shape = tuple(sample["image_shape"].tolist())
voxel_size = tuple(sample["voxel_size"].tolist())

image_shape, voxel_size

In [ ]:
%%time
unet_out, det_logits = model.encode(imgs_batch)

In [ ]:
bs, window_size, d, h, w = imgs_batch.shape
d_out, h_out, w_out = unet_out.shape[-3:]

unet_scaling = (d // d_out, h // h_out, w // w_out)

print(unet_scaling)
print(voxel_size)

In [ ]:
det_probas = imgs_batch[:, :, ::unet_scaling[0], ::unet_scaling[1], ::unet_scaling[2]]
det_probas.size()

In [ ]:
det_coords[:, 337], coords[:, i, :] / torch.tensor(unet_scaling).to(det_coords)

In [ ]:
frame_det = []

frame = 63

with torch.no_grad():
    det_probas = [det.sigmoid() for det in det_logits]
    for i in range(WINDOW_SIZE):         

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(sample["imgs"][i].squeeze()[frame].cpu().numpy())
        plt.scatter(
            coords[:, i, :, 1].cpu().numpy(),
            coords[:, i, :, 2].cpu().numpy(),
            c=None,
            edgecolor="r",
            marker="o",
            s=40,
        )
        plt.subplot(1, 2, 2)
        plt.imshow(det_probas[i].squeeze()[frame].cpu().numpy())
        plt.scatter(
            coords[:, i, :, 1].cpu().numpy() / unet_scaling[1],
            coords[:, i, :, 2].cpu().numpy() / unet_scaling[2],
            c=None,
            edgecolor="r",
            marker="o",
            s=40,
        )
        plt.colorbar()
        plt.show()

        det_coords, det_pos_embed, det_mask, matches = detect_and_match(
            det_probas[i],
            coords[:, i],
            masks[:, i],
            image_shape,
            voxel_size=voxel_size,
            unet_scaling=unet_scaling,
            pool_kernel_um=5,
            frame_index=i,
            window_size=WINDOW_SIZE,
            det_threshold=0.5
        )
        unet_feat = model._index_features(
            unet_out[:, i], det_coords, det_mask,
        )
        frame_det.append((det_coords, det_pos_embed, det_mask, matches, unet_feat))

        print("Pred peaks size:", det_coords.size())
        # print("Pred mask:", det_mask)
        if matches[0].max() > -1:
            print(f"Peak matches with gt {coords[:, i][int(matches[0].max())]}: {det_coords[0][int(matches[0].argmax())]}")
        else:
            print("No matches")

        # break

In [ ]:
edge_logits_list = []
edge_targets_list = []
det_mask_pairs = []

for i in range(WINDOW_SIZE - 1):
    ns = frame_det[i][0].shape[1]  # number of source nodes
    nt = frame_det[i + 1][0].shape[1]  # number of target nodes

    pair_target = build_matched_edge_targets(
        frame_det[i][3],  # matches at frame i
        frame_det[i + 1][3],  # matches at frame i+1
        targets[:, i],  # gt transition matrix at frame i
        ns,  # number of source nodes
        nt,  # number of target nodes
    )  # which node pairs are linked

    # Forward of the edge scorer - score each node pair
    edge_log = model.predict_edges(
        unet_feat_src=frame_det[i][4],  # frame i UNet features
        unet_feat_tgt=frame_det[i + 1][4],  # frame i+1 UNet features
        coords_src=frame_det[i][0],  # frame i coords
        coords_tgt=frame_det[i + 1][0],  # frame i+1 coords
        pos_feat_src=frame_det[i][1],  # frame i pos-embed
        pos_feat_tgt=frame_det[i + 1][1],  # frame i+1 pos-embed
        mask_src=frame_det[i][2],  # frame i mask
        mask_tgt=frame_det[i + 1][2],  # frame i+1 mask
    )
    edge_logits_list.append(edge_log)
    edge_targets_list.append(pair_target)
    det_mask_pairs.append((frame_det[i][2], frame_det[i + 1][2]))

### Loss

In [ ]:
from tracking.losses import TrackingLoss

In [ ]:
criterion = TrackingLoss(**{
        "det_loss_weight": 1.0,
        "det_neg_weight": 0.1,
        "det_weak_weight": 0.3,        # weight for image-derived weak positives
        "det_weak_smoothing": 0.1,     # smoothing factor for weak positives
    }
)

In [ ]:
%%time
loss, breakdown = criterion(
    det_logits,
    [coords[:, i] for i in range(window_size)],
    [masks[:, i] for i in range(window_size)],
    edge_logits_list,
    edge_targets_list,
    det_mask_pairs,
    imgs_per_frame=[imgs_batch[:, i] for i in range(window_size)],
    unet_scaling=unet_scaling,
)

In [ ]:
breakdown

Done !